# 3.4. Linear Regression Implementation from Scratch
D2L의 Linear Regression Implementation from Scratch 장을 PyTorch 기준으로 정리함.

이 장은 선형회귀 학습 루프를 직접 만드는 장이다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 목표

이전 장에서는 가짜 데이터를 만들었다

```py
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

X = torch.randn(2000, 2)
y = X @ true_w.reshape(-1, 1) + true_b + noise
```

이번 장의 목표는 모델이 진짜 값을 찾도록 하는 것이다.
처음 모델은 w, b를 모른다. 학습을 반복하면 이게 가까워져야한다.
D2L 원문도 synthetic data는 우리가 직접 만든 데이터라 true parameter를 정확히 알고 있고, 학습된 parameter와 true parameter를 비교해서 학습이 성공했는지 확인할 수 있다고 설명한다.

## 2. 전체 흐름

1. w, b를 랜덤하게 초기화한다.
2. X_batch를 모델에 넣어 y_hat을 만든다.
3. y_hat과 y_batch를 비교해서 loss를 계산한다.
4. loss.backward()로 gradient를 구한다.
5. SGD로 w, b를 업데이트한다.
6. gradient를 0으로 초기화한다.
7. 이 과정을 여러 epoch 반복한다.
8. 학습된 w, b가 true_w, true_b에 가까운지 확인한다.

학습 루프는 수식으로 이렇다
$$
w ← w - lr * dw \\
b ← b - lr * db
$$

dw와 db는 loss를 줄이기 위해서 w, b를 어디로 움직여야 할지 알려주는 gradient다.

## 3. 데이터 준비

이전 장처럼 데이터를 만든다.

In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# 진짜 정답 파라미터
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

# 데이터 개수
num_train = 1000
num_val = 1000
n = num_train + num_val

# feature 생성
X = torch.randn(n, len(true_w))

# noise 생성
noise = torch.randn(n, 1) * 0.01

# label 생성
y = X @ true_w.reshape(-1, 1) + true_b + noise

# train / validation 분리
X_train = X[:num_train]
y_train = y[:num_train]

X_val = X[num_train:]
y_val = y[num_train:]

# DataLoader 생성
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

torch.Size([1000, 2]) torch.Size([1000, 1])
torch.Size([1000, 2]) torch.Size([1000, 1])


## 4. 모델 파라미터 초기화

선형회귀 모델은 이렇다.

```py
y_hat = X @ w + b
```
여기서 학습해야 하는 것은 w, b이다.
ture_w, true_b는 데이터 만들 때 쓴 진짜 정답값이다. w,b는 처음에 모르기 때문에 랜덤으로 시작한다.

In [7]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True) # weight를 평균 0, 표준편차 0.01인 정규분포에서 뽑는다
b = torch.zeros(1, requires_grad=True) # gradient를 계산할 수 있도록 한다.

print("initial w:", w)
print("initial b:", b)

print("w.shape :", w.shape)
print("b.shape :", b.shape)

initial w: tensor([[-0.0107],
        [-0.0184]], requires_grad=True)
initial b: tensor([0.], requires_grad=True)
w.shape : torch.Size([2, 1])
b.shape : torch.Size([1])


## 5. 모델 정의

모델은 그냥 함수 하나이다.
```py
def linear_regression(X, w, b):
    return X @ w + b
```

D2L에서는 `forward` 메서드로 `torch.matmul(X, self.w) + self.b`를 정의한다. 원문도 입력 feature matrix `X`와 weight `w`의 matrix-vector product에 bias `b`를 더한다고 설명한다. b는 scalar라서 broadcasting으로 각 샘플에 더해진다.

In [9]:
def linear_regression(X, w, b):
    return X @ w + b

X_batch, y_batch = next(iter(train_loader))

y_hat = linear_regression(X_batch, w, b)

print("X_batch shape:", X_batch.shape)
print("w shape:", w.shape)
print("y_hat shape:", y_hat.shape)
print("y_batch shape:", y_batch.shape)

X_batch shape: torch.Size([32, 2])
w shape: torch.Size([2, 1])
y_hat shape: torch.Size([32, 1])
y_batch shape: torch.Size([32, 1])


## 6. Loss function 정의

선형회귀에서는 보통 squared loss를 쓴다.

```py
loss = (y_hat - y) ** 2 / 2
```
D2L도 이 절에서 squared loss를 쓰고, minibatch 안의 모든 example loss를 평균낸 값을 반환한다고 설명한다.

In [10]:
def squared_loss(y_hat, y):
    return ((y_hat - y) ** 2 / 2).mean()

## 7. SGD 구현

SGD는 Stochastic Gradient Descent이다.

```py
w = w - lr * w.grad
b = b - lr * b.grad
```
D2L도 minibatch에서 gradient를 추정하고, loss를 줄일 수 있는 방향으로 parameter를 업데이트한다고 설명한다. 또한 loss가 minibatch 평균으로 계산되기 때문에 지금 예제에서는 learning rate를 batch size에 맞춰 따로 조정하지 않아도 된다고 설명했다.

In [ ]:
def sgd(params, lr):
    with torch.no_grad(): # autograd가 추적하지 않기 위함.
        for param in params:
            param -= lr * param.grad # param.grad는 loss를 줄이기 위해 parameter를 어디로 움직여야하는지 알려줌.
            param.grad.zero_() # Reset gradients after each update.


## 8. 전체 학습 루프

D2L 원문도 딥러닝 모델에서도 비슷한 학습 루프를 계속 쓰기 때문에 training loop를 매우 중요하다고 말한다. 각 epoch에서 전체 training dataset을 한 번 지나가고, 각 iteration마다 minibatch를 꺼내 loss를 계산하고, gradient를 계산한 뒤 optimizer로 parameter를 업데이트한다고 설명한다.

In [20]:
# 1. 파라미터 초기화
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 2. 하이퍼파라미터
lr = 0.03
num_epochs = 5

# 3. 학습 루프
for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:
        # forward
        y_hat = linear_regression(X_batch, w, b)

        # loss 계산
        loss = squared_loss(y_hat, y_batch)

        # backward
        loss.backward()

        # parameter update
        sgd([w, b], lr)

    # epoch마다 train loss 확인
    with torch.no_grad():
        train_loss = squared_loss(linear_regression(X_train, w, b), y_train)
        val_loss = squared_loss(linear_regression(X_val, w, b), y_val)

    print(f"epoch {epoch + 1}, train loss {train_loss.item():.6f}, val loss {val_loss.item():.6f}")

epoch 1, train loss 2.413657, val loss 2.392542
epoch 2, train loss 0.385703, val loss 0.384640
epoch 3, train loss 0.060090, val loss 0.060361
epoch 4, train loss 0.009623, val loss 0.009727
epoch 5, train loss 0.001573, val loss 0.001597


## 9. 학습 후 파라미터 확인

정답은 이거였다.
```py
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
```

label에 noise가 있기 때문에 완전히 똑같지 않을 수 있다. epoch도 5번인데 더 돌린다면 가까워질 수 있다.

D2L에서는 true parameter를 정확히 복구하는 것을 당연하게 여기면 안 된다고 말한다. 머신러닝에서는 진짜 underlying parameter를 정확히 복구하는 것보다, 예측을 잘하는 parameter를 찾는 것이 더 중요한 경우가 많다.

In [21]:
print("learned w:", w.reshape(-1))
print("true w:", true_w)
print("w error:", true_w - w.reshape(-1))

print("learned b:", b.item())
print("true b:", true_b)
print("b error:", true_b - b.item())

learned w: tensor([ 1.9698, -3.3683], grad_fn=<ViewBackward0>)
true w: tensor([ 2.0000, -3.4000])
w error: tensor([ 0.0302, -0.0317], grad_fn=<SubBackward0>)
learned b: 4.163679122924805
true b: 4.2
b error: 0.03632087707519549


## 10. 오늘의 정리

- 이번 절은 선형회귀 모델을 scratch로 직접 구현하는 장이다.
- D2L의 Module, Trainer, add_to_class는 지금 단계에서 중요하지 않다.
- 중요한 것은 w, b, y_hat, loss, backward, SGD update의 흐름이다.
- w와 b는 모델이 학습해야 하는 파라미터다.
- true_w와 true_b는 데이터를 만들 때 사용한 진짜 정답이다.
- 모델 예측은 y_hat = X @ w + b 로 계산한다.
- loss는 예측값 y_hat과 정답 y의 차이를 측정한다.
- squared loss는 ((y_hat - y) ** 2 / 2).mean() 으로 구현할 수 있다.
- loss.backward()는 w.grad와 b.grad를 계산한다.
- backward() 자체는 w, b를 바꾸지 않는다.
- SGD update가 실제로 w, b를 바꾼다.
- 업데이트 공식은 param = param - lr * param.grad 이다.
- gradient는 PyTorch에서 누적되므로 매번 zero_()로 초기화해야 한다.
- train_loader는 학습에 사용되고, val_loader는 검증에 사용된다.
- train에서는 w, b를 업데이트하지만 validation에서는 업데이트하지 않는다.
- epoch은 train dataset 전체를 한 번 다 보는 것이다.
- batch size 32라면 한 번에 32개 샘플씩 모델에 넣는다.